Cedar will meet both the capacity and budget requirements.

In [1]:
from copy import deepcopy

rooms = {
    "Juniper": {"capacity": 12, "cost_cents": 4000},
    "Mesa": {"capacity": 20, "cost_cents": 6500},
    "Cedar": {"capacity": 18, "cost_cents": 5000},
}

request = {"id": "evt-1", "people": 16, "budget_cents": 6000}

for name, details in rooms.items():
    print(name, details)

print(request)

Juniper {'capacity': 12, 'cost_cents': 4000}
Mesa {'capacity': 20, 'cost_cents': 6500}
Cedar {'capacity': 18, 'cost_cents': 5000}
{'id': 'evt-1', 'people': 16, 'budget_cents': 6000}


Cedar meets both requirements. Juniper is too small, and Mesa is over budget.

In [2]:
def capacity_worker(payload, catalog):
    return sorted(name for name, r in catalog.items()
                  if r["capacity"] >= payload["people"])

def budget_worker(payload, catalog):
    return sorted(name for name, r in catalog.items()
                  if r["cost_cents"] <= payload["budget_cents"])

print("Capacity:", capacity_worker({"people": 16}, rooms))
print("Budget:", budget_worker({"budget_cents": 6000}, rooms))

Capacity: ['Cedar', 'Mesa']
Budget: ['Cedar', 'Juniper']


Each worker checks a different requirement. Both are needed to find the right room. A reply from a different request could cause incorrect results.

In [3]:
workers = {"capacity": capacity_worker, "budget": budget_worker}

messages = [
    {"request_id": request["id"], "to": "capacity",
     "payload": {"people": request["people"]}},
    {"request_id": request["id"], "to": "budget",
     "payload": {"budget_cents": request["budget_cents"]}},
]

for message in messages:
    print(message)

{'request_id': 'evt-1', 'to': 'capacity', 'payload': {'people': 16}}
{'request_id': 'evt-1', 'to': 'budget', 'payload': {'budget_cents': 6000}}


Both workers use the same request ID and only receive the information they need. Neither worker should run if the message budget is only one.

In [4]:
def dispatch(messages, catalog, max_messages=2):
    if type(max_messages) is not int or max_messages < 1:
        raise ValueError("positive integer budget required")
    if len(messages) > max_messages:
        raise ValueError("message budget exceeded")

    replies = []

    for message in messages:
        role = message["to"]
        if role not in workers:
            raise ValueError("unknown worker")

        candidates = workers[role](deepcopy(message["payload"]),
                                   deepcopy(catalog))

        replies.append({"request_id": message["request_id"],
                        "from": role, "status": "ok",
                        "candidates": candidates})

    return replies

replies = dispatch(messages, rooms)
print(replies)

[{'request_id': 'evt-1', 'from': 'capacity', 'status': 'ok', 'candidates': ['Cedar', 'Mesa']}, {'request_id': 'evt-1', 'from': 'budget', 'status': 'ok', 'candidates': ['Cedar', 'Juniper']}]


Both workers returned their results. The message budget limits how much work can be dispatched. The coordinator should double-check the results to avoid incorrect recommendations.

In [5]:
def combine(replies, request, catalog):
    accepted = {}

    for reply in replies:
        role = reply["from"]

        if reply["request_id"] != request["id"]:
            raise ValueError("stale reply")

        if role not in workers or role in accepted or reply["status"] != "ok":
            raise ValueError("missing, failed or duplicate role")

        values = reply["candidates"]

        if type(values) is not list or any(v not in catalog for v in values):
            raise ValueError("invalid candidates")

        accepted[role] = set(values)

    if set(accepted) != set(workers):
        raise ValueError("incomplete specialist set")

    common = accepted["capacity"] & accepted["budget"]

    return sorted(name for name in common
                  if catalog[name]["capacity"] >= request["people"]
                  and catalog[name]["cost_cents"] <= request["budget_cents"])

print("Joint recommendation:", combine(replies, request, rooms))

Joint recommendation: ['Cedar']


Cedar meets both requirements. The coordinator checks the original data to make sure the recommendation is correct. An old request ID should be rejected.

In [6]:
stale = deepcopy(replies)
stale[0]["request_id"] = "evt-old"

for label, sample in [("stale", stale), ("missing", replies[:1])]:
    try:
        combine(sample, request, rooms)
    except ValueError as error:
        print(label, ":", error)

print("Original request ID:", replies[0]["request_id"])

stale : stale reply
missing : incomplete specialist set
Original request ID: evt-1


The coordinator rejects outdated and incomplete replies without changing the original data. No room will meet the requirements for 19 people.

In [7]:
def coordinate(people, budget, request_id):
    if (type(people) is not int or people < 1
            or type(budget) is not int or budget < 0):
        raise ValueError("invalid request quantities")

    task = {"id": request_id, "people": people, "budget_cents": budget}

    batch = [
        {"request_id": request_id, "to": "capacity",
         "payload": {"people": people}},
        {"request_id": request_id, "to": "budget",
         "payload": {"budget_cents": budget}},
    ]

    answers = dispatch(batch, rooms)
    return combine(answers, task, rooms)

print("19 people:", coordinate(19, 6000, "evt-2"))
print("12 people:", coordinate(12, 6000, "evt-3"))

19 people: []
12 people: ['Cedar', 'Juniper']


No rooms meet the requirements for 19 people, but two rooms work for 12 people. Nothing is actually booked.  Parallel execution can save time without reducing the total work.

In [8]:
worker_times = [3, 5]
coordination = 2

sequential_elapsed = sum(worker_times) + coordination
ideal_parallel_elapsed = max(worker_times) + coordination
work = sum(worker_times) + coordination

print("Sequential model:", sequential_elapsed)
print("Ideal parallel model:", ideal_parallel_elapsed)
print("Total modeled work:", work)
print("Booking performed:", False)

Sequential model: 10
Ideal parallel model: 7
Total modeled work: 10
Booking performed: False


Parallel execution could save time, but the total work stays the same. These times are estimates, not actual measurements. The direct function should return the same results as the coordinator.

In [9]:
def direct_check(people, budget, catalog):
    return sorted(name for name, r in catalog.items()
                  if r["capacity"] >= people
                  and r["cost_cents"] <= budget)

for people in [16, 19, 12]:
    print("People:", people)
    print("Direct:", direct_check(people, 6000, rooms))
    print("Coordinator:", coordinate(people, 6000, "evt-test"))

corrupted = deepcopy(replies)

for reply in corrupted:
    reply["candidates"] = ["Mesa"]

print("Corrupted result:", combine(corrupted, request, rooms))

People: 16
Direct: ['Cedar']
Coordinator: ['Cedar']
People: 19
Direct: []
Coordinator: []
People: 12
Direct: ['Cedar', 'Juniper']
Coordinator: ['Cedar', 'Juniper']
Corrupted result: []


Both methods returned the same results. Mesa was rejected because it was over budget. One limitation is that if a worker leaves out Cedar, the coordinator could miss a valid room.